# Téléchargement d'images avec Google Earth Engine

## Introduction

Ce notebook met en place une chaîne de traitement complète pour télécharger des images satellites **Landsat** (Collection 2, niveau Surface Reflectance) via **Google Earth Engine (GEE)**, sur une zone d'étude définie par l'utilisateur.

L'objectif est de produire, pour plusieurs années clés (1985, 1995, 2005, 2015, 2025), un **composite annuel médian**, nettoyé des nuages et de leurs ombres, corrigé radiométriquement, découpé sur la zone d'étude (ROI), puis exporté localement au format `.tif`. Ces composites pourront ensuite être utilisés pour des analyses temporelles (changement d'occupation du sol, indices spectraux, etc.).

Les grandes étapes couvertes dans ce notebook sont les suivantes :

1. **Initialisation** de l'environnement (librairies, connexion à Google Earth Engine, chemins d'entrée/sortie).
2. **Chargement de la zone d'étude** (ROI) et conversion en objet Earth Engine.
3. **Compréhension du masquage des nuages** via la bande `QA_PIXEL` et des opérations sur les bits.
4. **Correction radiométrique** des images Landsat (passage des valeurs brutes à la réflectance de surface).
5. **Harmonisation des bandes spectrales** entre les capteurs Landsat 5, 7 et 8 à l'aide d'un dictionnaire de configuration.
6. **Téléchargement automatisé** des composites annuels pour plusieurs années et plusieurs satellites.
7. **Visualisation** des résultats sur une carte interactive, en compositions RGB et NRG.

Ce document combine du code Python et des explications pédagogiques détaillées, afin que chaque étape du traitement (du bit `QA_PIXEL` jusqu'à l'export du raster final) soit compréhensible, même pour quelqu'un découvrant la télédétection.


## Librairies

Ce bloc importe les bibliothèques nécessaires pour :
- accéder à Google Earth Engine avec `ee`,
- manipuler les fichiers locaux et les chemins avec `os`,
- afficher les données géospatiales dans une carte interactive avec `geemap`,
- lire et manipuler la zone d'étude vectorielle avec `geopandas`.

In [ ]:
import os
import ee
import geemap
import geopandas as gpd

# Pas nécessaire pour vous
from config import GEE_PROJECT

## Initialiser GEE et geemap

Ce bloc initialise l'accès à Google Earth Engine avec le projet GEE souhaité. Cela permet de se connecter à votre compte GEE et d'exécuter ensuite des requêtes sur les collections d'images et la zone d'étude.

La fonction `ee.Initialize(project='gee-proj1-468022')` établit la configuration du projet, ce qui est nécessaire pour utiliser `ee.ImageCollection`, `ee.Geometry` et les autres objets Earth Engine dans le notebook.

In [ ]:
ee.Initialize(project=GEE_PROJECT)

## Définir les entrées et sorties (I/I)

Ce bloc fixe les chemins des fichiers utilisés par le notebook :
- `roi_path` : le fichier de la zone d'étude vectorielle (ici un GeoPackage contenant le polygone ROI),
- `img_out_dir` : le dossier local où seront enregistrées les images Landsat exportées.

Ces variables permettent de séparer facilement la configuration des données d'entrée et de sortie du reste du traitement.

In [7]:
zone_detude = 'data/zone_detude/zone_detude.gpkg'
img_landsat = 'data/images_landsat'

## Ajouter la zone d'étude

Dans cette section, nous chargeons le polygone de la zone d'étude depuis le fichier GeoPackage et le convertissons en un objet Earth Engine. Cela permet de :
- visualiser la zone d'étude sur la carte,
- filtrer les images Landsat sur cette zone,
- découper les composites exportés exactement sur le contour du ROI.

In [11]:
ROI = gpd.read_file(zone_detude)
ROI_ee = geemap.gdf_to_ee(ROI)
ROI_ee_styled = ROI_ee.style(color='red', fillColor='00000000', width=2)

m = geemap.Map(zoom=5)
m.center_object(ROI_ee, zoom=12)
m.addLayer(ROI_ee_styled, {}, 'ROI')
m

Map(center=[16.039745167717093, -16.45725266033479], controls=(WidgetControl(options=['position', 'transparent…

<img src="images/roi_gee.png" width="2000">

## Télécharger les images landsat

### Masquage des nuages avec `QA_PIXEL` — Landsat Collection 2

Les images satellitaires Landsat contiennent souvent des **nuages et leurs ombres**.
Ces pixels sont inutilisables pour l'analyse de la surface terrestre car ils ne reflètent pas la réalité du sol.

Sans masquage, un composite médian annuel contiendra des artefacts nuageux qui fausseront :
- les indices spectraux (NDVI, NDWI...)
- la classification de l'occupation du sol
- toute analyse temporelle

Landsat Collection 2 fournit une bande **`QA_PIXEL`** (Quality Assessment) qui encode la qualité de chaque pixel sous forme de **bits**. Chaque pixel de `QA_PIXEL` est un entier 16 bits. Un **bit** est la plus petite unité d'information en informatique. Il ne peut prendre que deux valeurs : **0** ou **1** — comme un interrupteur éteint ou allumé.

En mémoire, les nombres sont stockés comme une suite de bits. Par exemple le nombre **42** s'écrit en binaire sur 8 bits :

```
Position :   7    6    5    4    3    2    1    0
Valeur   :   0    0    1    0    1    0    1    0
             ↑                        ↑         ↑
           bit7                     bit3      bit0
```

Chaque position a une valeur qui est une puissance de 2 :

```
bit0 = 2⁰ =   1
bit1 = 2¹ =   2
bit2 = 2² =   4
bit3 = 2³ =   8
bit4 = 2⁴ =  16
bit5 = 2⁵ =  32
bit6 = 2⁶ =  64
bit7 = 2⁷ = 128
```

Donc 42 = 32 + 8 + 2 = bit5 + bit3 + bit1 tous à 1.

**Pourquoi stocker plusieurs infos dans un seul nombre ?**

C'est une technique très courante pour **compresser plusieurs questions oui/non** dans un seul entier. Landsat utilise ça dans `QA_PIXEL` : plutôt que de stocker 16 colonnes séparées (nuage, ombre, neige...), tout tient dans **un seul entier de 16 bits** — un bit par question.

```
QA_PIXEL d'un pixel = 0000  0000     0010    1000
                                       ↑     ↑
                                     bit5    bit3
                                    nuage    ombre
```

**`1 << 5` : comment on pointe vers un bit précis ?**

`<<` est l'opérateur de **décalage à gauche** : il déplace le chiffre 1 vers la position voulue.

```
1 << 0  →  0000 0001  =   1   (pointe sur bit 0)
1 << 1  →  0000 0010  =   2   (pointe sur bit 1)
1 << 3  →  0000 1000  =   8   (pointe sur bit 3 = ombre)
1 << 5  →  0010 0000  =  32   (pointe sur bit 5 = nuage)
```

C'est un **projecteur** : il illumine uniquement le bit qui nous intéresse, tous les autres restent à 0.

### Correction radiométrique des images Landsat

Le satellite enregistre d’abord un signal brut appelé `Digital Number (DN`). Le `DN` est une valeur entière issue du capteur (généralement sur 16 bits, donc entre 0 et 65535).
Il ne représente ni une température, ni une énergie, ni une réflectance : il s’agit uniquement d’un signal électronique brut. La `calibration radiométrique` permet de transformer ces valeurs `DN` en grandeurs physiques interprétables (radiance ou réflectance).

Dans le cas de `Landsat Collection 2 Level-2`, les produits fournis correspondent déjà à de la `Surface Reflectance (SR)`, c’est-à-dire une réflectance corrigée des effets atmosphériques. Cependant, ces valeurs ne sont pas stockées directement en flottants, mais sous forme de valeurs entières scalées pour optimiser le stockage et la transmission des données. l est donc nécessaire d’appliquer une transformation linéaire pour retrouver la réflectance physique :

`Réflectance = DN × scale + offset`

Le `scale` est un multiplicateur qui sert à réduire ou ajuster l’amplitude des valeurs. Le `offsetf` est une constante ajoutée ou soustraite, servant à corriger un biais du capteur.

### Téléchargement

#### Description des bandes Landsat 5 / 7 / 8

Les satellites Landsat mesurent le rayonnement électromagnétique réfléchi ou émis par la surface terrestre dans plusieurs **intervalles de longueurs d'onde** appelés bandes spectrales. Chaque bande capte une portion différente du spectre et révèle des informations spécifiques sur le sol, la végétation, l'eau ou l'atmosphère.

---

**Landsat 5 — TM (Thematic Mapper) · 1984-2013**

| Bande GEE | Nom | Longueur d'onde | Résolution | Utilité principale |
|-----------|-----|-----------------|------------|--------------------|
| `SR_B1` | Bleu | 0.45–0.52 µm | 30 m | Eau, atmosphère, bathymétrie côtière |
| `SR_B2` | Vert | 0.52–0.60 µm | 30 m | Végétation verte, turbidité de l'eau |
| `SR_B3` | Rouge | 0.63–0.69 µm | 30 m | Absorption chlorophylle, distinction végétation/sol |
| `SR_B4` | Proche infrarouge (NIR) | 0.76–0.90 µm | 30 m | Biomasse végétale, délimitation eau/terre |
| `SR_B5` | SWIR 1 | 1.55–1.75 µm | 30 m | Humidité sol et végétation, minéraux |
| `SR_B6` | Thermique (TIR) | 10.40–12.50 µm | 120 m | Température de surface |
| `SR_B7` | SWIR 2 | 2.08–2.35 µm | 30 m | Géologie, minéraux argileux, zones brûlées |

---

**Landsat 7 — ETM+ (Enhanced Thematic Mapper Plus) · 1999-2022**

Même configuration que Landsat 5 avec l'ajout d'une bande panchromatique haute résolution.

| Bande GEE | Nom | Longueur d'onde | Résolution | Utilité principale |
|-----------|-----|-----------------|------------|--------------------|
| `SR_B1` | Bleu | 0.45–0.52 µm | 30 m | Eau, atmosphère |
| `SR_B2` | Vert | 0.52–0.60 µm | 30 m | Végétation, turbidité |
| `SR_B3` | Rouge | 0.63–0.69 µm | 30 m | Chlorophylle, sol nu |
| `SR_B4` | NIR | 0.77–0.90 µm | 30 m | Biomasse, eau/terre |
| `SR_B5` | SWIR 1 | 1.55–1.75 µm | 30 m | Humidité, minéraux |
| `SR_B6_VCID_1` | Thermique (bas gain) | 10.40–12.50 µm | 60 m | Température de surface |
| `SR_B7` | SWIR 2 | 2.09–2.35 µm | 30 m | Géologie, zones brûlées |
| `SR_B8` | Panchromatique | 0.52–0.90 µm | **15 m** | Fusion d'images (pansharpening) |

</br>

> ⚠️ Depuis mai 2003, le **Scan Line Corrector (SLC)** de Landsat 7 est en panne. Les images post-2003 présentent des **bandes noires** (données manquantes) sur les bords de chaque scène, couvrant environ 22% de la surface.

---

**Landsat 8 — OLI/TIRS (Operational Land Imager) · 2013-présent**

Landsat 8 apporte deux nouvelles bandes (Coastal/Aerosol et Cirrus) et sépare le thermique en deux canaux.

| Bande GEE | Nom | Longueur d'onde | Résolution | Utilité principale |
|-----------|-----|-----------------|------------|--------------------|
| `SR_B1` | Coastal / Aerosol | 0.43–0.45 µm | 30 m | Zones côtières, aérosols atmosphériques |
| `SR_B2` | Bleu | 0.45–0.51 µm | 30 m | Eau, atmosphère |
| `SR_B3` | Vert | 0.53–0.59 µm | 30 m | Végétation, turbidité |
| `SR_B4` | Rouge | 0.64–0.67 µm | 30 m | Chlorophylle, sol nu |
| `SR_B5` | NIR | 0.85–0.88 µm | 30 m | Biomasse, eau/terre |
| `SR_B6` | SWIR 1 | 1.57–1.65 µm | 30 m | Humidité sol/végétation |
| `SR_B7` | SWIR 2 | 2.11–2.29 µm | 30 m | Géologie, zones brûlées |
| `SR_B8` | Panchromatique | 0.50–0.68 µm | **15 m** | Pansharpening |
| `SR_B9` | Cirrus | 1.36–1.38 µm | 30 m | Détection des nuages cirrus |
| `ST_B10` | Thermique (TIRS 1) | 10.60–11.19 µm | 100 m | Température de surface |

---

**Comparaison des bandes entre satellites**

Les bandes équivalentes n'ont **pas le même numéro** selon le satellite :

| Rôle | Landsat 5 | Landsat 7 | Landsat 8 |
|------|-----------|-----------|----------|
| Bleu | B1 | B1 | **B2** |
| Vert | B2 | B2 | **B3** |
| Rouge | B3 | B3 | **B4** |
| NIR | B4 | B4 | **B5** |
| SWIR 1 | B5 | B5 | **B6** |
| SWIR 2 | B7 | B7 | **B7** |
| Thermique | B6 | B6_VCID_1 | **ST_B10** |

---

**Compositions courantes**

| Composition | Bandes | Ce qu'on voit |
|-------------|----------------|---------------|
| **RGB** (vraies couleurs) | R-G-B | Aspect naturel, proche de ce que l'œil voit |
| **NRG** (fausse couleur) | NIR-R-G | Végétation en rouge vif, eau sombre, sols nus beiges |
| **SWIR** (zones brûlées) | SWIR2-NIR-R | Zones brûlées en rouge/orange, végétation verte |
| **Géologie** | SWIR2-SWIR1-R | Différenciation lithologique, minéraux argileux |

#### Configuration des bandes

Dans le code ci-dessous, un dictionnaire de configuration est utilisé pour harmoniser le traitement des images Landsat 5, 7 et 8 issues de la collection Surface Reflectance Level-2 de Google Earth Engine. Chaque capteur possède sa propre organisation de bandes spectrales (bleu, vert, rouge, proche infrarouge et SWIR), ce qui explique les différences de nomenclature entre SR_B1, SR_B2, etc. Le dictionnaire permet donc d’unifier ces différences en associant des noms standards (B, G, R, N, S1, S2) aux bandes correspondantes selon le satellite. Les images utilisées proviennent des collections officielles Landsat accessibles dans Google Earth Engine, par exemple Landsat 5 : [https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LT05_C02_T1_L2](https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LT05_C02_T1_L2), Landsat 7 : [https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LE07_C02_T1_L2](https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LE07_C02_T1_L2), et Landsat 8 : [https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC08_C02_T1_L2](https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC08_C02_T1_L2). Enfin, les paramètres scale_factor`` et offset`` permettent de convertir les valeurs stockées (entiers scalés pour optimisation mémoire) en surface reflectance physique réelle, selon une transformation linéaire de type `réflectance = DN × scale + offset`, garantissant ainsi la comparabilité temporelle et inter-capteurs des données.

In [12]:
config = {
    '05': {
        "collection": "LANDSAT/LT05/C02/T1_L2",
        "bands":      {"B": "SR_B1", "G": "SR_B2", "R": "SR_B3",
                       "N": "SR_B4", "S1": "SR_B5", "S2": "SR_B7"},
        "scale_factor": 0.0000275,
        "offset":       -0.2,
    },
    '07': {
        "collection": "LANDSAT/LE07/C02/T1_L2",
        "bands":      {"B": "SR_B1", "G": "SR_B2", "R": "SR_B3",
                       "N": "SR_B4", "S1": "SR_B5", "S2": "SR_B7"},
        "scale_factor": 0.0000275,
        "offset":       -0.2,
    },
    '08': {
        "collection": "LANDSAT/LC08/C02/T1_L2",
        "bands":      {"B0": "SR_B1", "B": "SR_B2", "G": "SR_B3", "R": "SR_B4",
                       "N": "SR_B5", "S1": "SR_B6", "S2": "SR_B7"},
        "scale_factor": 0.0000275,
        "offset":       -0.2,
    },
}

#### Fonction de téléchargement d'une image composite

Cette fonction automatise toute la chaîne de traitement pour obtenir une image Landsat **propre, corrigée et découpée** sur une zone d'étude :

1. Elle interroge Google Earth Engine pour récupérer toutes les images Landsat d'une année donnée sur la zone
2. Elle masque les nuages et ombres de nuages
3. Elle applique la correction radiométrique (valeurs brutes → réflectance)
4. Elle calcule un composite médian (une seule image représentative de l'année)
6. Elle télécharge l'image en local au format `.tif`

Sans cette fonction, chacune de ces étapes demanderait une dizaine de lignes de code séparées.

| Paramètre  | Type         | Obligatoire | Description                                                                 |
|------------|--------------|-------------|-----------------------------------------------------------------------------|
| `roi_ee`   | `ee.Geometry`| oui         | Zone d'étude — toutes les images sont filtrées et découpées sur cette zone |
| `year`     | `int`        | oui         | Année cible (ex: `1985`). La fonction prend les images du 01-01 au 31-12  |
| `bands`    | `list` | non     | Bandes à exporter (ex: `['SR_B1','SR_B2']`). Si `None`, toutes les bandes optiques sont utilisées |
| `satellite`| `int`        | non (défaut `5`) | Numéro du satellite Landsat : `5`, `7` ou `8` — choisit la collection et le mapping des bandes |
| `output_dir`| `str`       | non (défaut `'.'`) | Dossier de destination du fichier `.tif`                                   |
| `filename` | `str/None` | non     | Nom du fichier de sortie. Si `None`, généré automatiquement : `L{satellite}_{year}.tif` |


**Ce que retourne la fonction**

- Un objet `ee.Image` (le composite GEE) — utilisable directement pour d'autres calculs GEE
- En effet de bord : le fichier `.tif` est écrit sur le disque et la carte `m` est mise à jour avec deux calques (RGB et NRG)

**Étape 1 : Lecture de la configuration satellite**

```python
cfg      = config[satellite]
band_map = cfg["bands"]     # correspondance nom logique → nom GEE
scale_f  = cfg["scale_factor"]  # facteur de correction radiométrique
offset   = cfg["offset"]        # décalage de correction radiométrique
```

Chaque satellite a ses propres noms de bandes dans GEE et ses propres coefficients de correction.  
Le dictionnaire `config` centralise tout ça pour éviter de dupliquer le code.

---

**Étape 2 : Filtrage de la collection**

```python
image_col
  .filterBounds(roi_ee)              # garder uniquement les images qui couvrent la zone
  .filterDate(start_date, end_date)  # garder uniquement les images de l'année
  .limit(5)                          # limiter à 5 images pour accélérer les tests
```

GEE contient des milliers d'images Landsat — sans filtre, le calcul serait impossible.

---

**Étape 3 : Masquage des nuages (`mask_clouds`)**

```python
.map(mask_clouds)
```

Applique la fonction `mask_clouds` à chaque image de la collection.  
Les pixels nuageux ou ombragés sont mis à `NaN` — ils ne participeront pas au composite.  
*(Voir la section suivante pour le détail de cette fonction.)*

---

**Étape 4 : Correction radiométrique (`apply_scale`)**

```python
.map(apply_scale)
```

Les valeurs brutes Landsat (entiers 0–65535) sont converties en **réflectance de surface** (flottants 0.0–1.0) :

```
réflectance = valeur_brute × 0.0000275 + (−0.2)
```

Sans cette correction, les valeurs ne sont pas comparables entre images ni entre satellites.

---

**Étape 5 : Composite médian**

```python
.median()
```

Pour chaque pixel, on prend la **valeur médiane** parmi toutes les images valides de l'année.  
La médiane est robuste aux valeurs extrêmes (nuages résiduels, anomalies) — elle donne une image représentative du sol sans artefacts.

```
Image janv.  : [0.12, NaN,  0.18, 0.15]
Image avril  : [0.14, 0.11, NaN,  0.16]
Image août   : [0.13, 0.10, 0.17, NaN ]
→ Médiane    : [0.13, 0.10, 0.17, 0.15]  ← NaN ignorés
```

---

**Étape 6 : Découpage et visualisation**

```python
.clip(roi_ee)         # découper sur la zone d'étude exacte
m.addLayer(...)       # afficher RGB dans la carte
m.addLayer(...)       # afficher NRG (fausse couleur) dans la carte
```

Deux calques sont ajoutés à la carte `m` : la composition RGB (vraies couleurs) et NRG (fausse couleur qui fait ressortir la végétation en rouge).

---

**Étape 7 : Export local**

```python
geemap.ee_export_image(
    composite,
    filename      = output_path,
    scale         = 30,          # résolution 30m (native Landsat)
    region        = roi_ee,      # découpage final
    crs           = "EPSG:3857", # projection Web Mercator
    file_per_band = False        # un seul .tif multi-bandes
)
```

`geemap.ee_export_image` télécharge l'image directement depuis GEE en local, sans passer par Google Drive. La résolution de 30m est la résolution native des capteurs Landsat.


**Exemple d'utilisation**

```python
# Landsat 5, année 1985, bandes optiques (SR) 1-5 et 7
bands_1985 = list(config['05']['bands'].values())
composite = download_landsat_composite(
    roi_ee     = ROI_ee,
    year       = 1985,
    bands      = bands_1985,
    satellite  = 5,
    output_dir = 'outputs/images',
)
# → écrit  outputs/images/L5_1985.tif
# → ajoute deux calques dans la carte m
# → retourne l'ee.Image pour la suite du traitement
```


**Code source de la fonction**


In [ ]:
def download_landsat_composite(roi_ee, year, bands=None, satellite='05', output_dir=".", filename=None):
    
    if satellite not in config:
        raise ValueError("satellite doit être 5, 7 ou 8")

    cfg      = config[satellite]
    band_map = cfg["bands"]
    scale_f  = cfg["scale_factor"]
    offset   = cfg["offset"]
    start_date = f'{year}-01-01'
    end_date = f'{year+1}-01-01'

    if bands is None:
        bands = list(band_map.values())

    def mask_clouds(image):
        qa   = image.select("QA_PIXEL")
        mask = qa.bitwiseAnd(1 << 3).eq(0).\
                    And(qa.bitwiseAnd(1 << 5).eq(0))
        return image.updateMask(mask)

    def apply_scale(image):
        optical = image.select("SR_B.*").multiply(scale_f).add(offset)
        return image.addBands(optical, overwrite=True)
    
    image_col = ee.ImageCollection(cfg["collection"])

    composite = (image_col
                 .filterBounds(roi_ee)
                 .filterDate(start_date, end_date)
                 .map(mask_clouds)
                 .map(apply_scale)
                 .select(bands)
                 .median()
                 .select(list(band_map.values()),
                         list(band_map.keys()))
                 .clip(roi_ee)
    )

    if filename is None:
        filename = f"L{satellite}_{year}.tif"

    output_path = os.path.join(output_dir, filename)
    os.makedirs(output_dir, exist_ok=True)
    
    geemap.ee_export_image(
        composite,
        filename      = output_path,
        scale         = 30,
        region        = roi_ee,
        crs           = "EPSG:3857",
        file_per_band = False
    )

    if os.path.isfile(output_path):
        print(f"Enregistré : {output_path}")

    return composite

#### Téléchargement automaisé pour 1985-1995-2005-2015-2025

Le bloc ci-dessous automatise les téléchargements pour plusieurs années clés en sélectionnant le bon satellite Landsat pour chaque période. Il effectue :

- la sélection du satellite (`05`, `07`, `08`) selon l'année,
- la construction du nom de fichier de sortie `L{satellite}_{year}.tif`,
- l'extraction des bandes optiques du dictionnaire `config`,
- l'appel à `download_landsat_composite(...)` pour chaque année,
- et la sauvegarde locale du raster généré.

Cette étape permet d'obtenir rapidement des composites annuels cohérents pour 1985, 1995, 2005, 2015 et 2025.

In [15]:
years = [1985, 1995, 2005, 2015, 2025]
satellite_by_year = {
    1985: '05',
    1995: '05',
    2005: '07',
    2015: '08',
    2025: '08',
}

results = {}

for year in years:
    satellite = satellite_by_year[year]
    filename = f"L{satellite}_{year}.tif"
    bands = list(config[satellite]["bands"].values())
    print(f"\n=== Téléchargement Landsat {satellite} - {year} ===")
    
    results[year] = download_landsat_composite(
        roi_ee=ROI_ee.geometry(),
        year=year,
        bands=bands,
        satellite=satellite,
        output_dir=img_landsat,
        filename=filename,
    )



=== Téléchargement Landsat 05 - 1985 ===
Generating URL ...
Please wait ...
Data downloaded to /home/bachir/Documents/From-HP-Desktop/Courses/TD-L3-GEO/td-l3geo-online/data/images_landsat/L05_1985.tif
Enregistré : data/images_landsat/L05_1985.tif

=== Téléchargement Landsat 05 - 1995 ===
Generating URL ...
Please wait ...
Data downloaded to /home/bachir/Documents/From-HP-Desktop/Courses/TD-L3-GEO/td-l3geo-online/data/images_landsat/L05_1995.tif
Enregistré : data/images_landsat/L05_1995.tif

=== Téléchargement Landsat 07 - 2005 ===
Generating URL ...
Please wait ...
Data downloaded to /home/bachir/Documents/From-HP-Desktop/Courses/TD-L3-GEO/td-l3geo-online/data/images_landsat/L07_2005.tif
Enregistré : data/images_landsat/L07_2005.tif

=== Téléchargement Landsat 08 - 2015 ===
Generating URL ...
Please wait ...
Data downloaded to /home/bachir/Documents/From-HP-Desktop/Courses/TD-L3-GEO/td-l3geo-online/data/images_landsat/L08_2015.tif
Enregistré : data/images_landsat/L08_2015.tif

=== Tél

#### Visualisation des composites annuels

Ce bloc ajoute chaque composite annuel à la carte interactive sous deux compositions spectrales :

- **RGB** (vraies couleurs) — aspect naturel, proche de ce que l'œil voit
- **NRG** (fausse couleur) — végétation en rouge vif, eau sombre, sols nus beiges

### Déroulement de la boucle

Pour chaque année du dictionnaire `results` :

1. On vérifie que le composite existe (`comp is not None`) — une année sans image valide est ignorée
2. On récupère le satellite correspondant via `satellite_by_year` — nécessaire pour lire les bons noms de bandes
3. On lit les noms de bandes depuis `config` — car le Rouge n'a pas le même numéro selon le satellite (`SR_B3` sur Landsat 5, `SR_B4` sur Landsat 8)
4. On ajoute deux calques à la carte `m` avec leurs paramètres de visualisation

#### Paramètres de visualisation

| Paramètre | Valeur | Signification |
|-----------|--------|---------------|
| `min` | `0.0` | Réflectance minimale affichée (noir) |
| `max` | `0.5` | Réflectance maximale affichée (blanc) — valeur "typique" des surfaces terrestres après correction radiométrique |
| `gamma` | `1.2` | Correction de luminosité — légèrement éclairci pour améliorer le contraste visuel |

En fin de boucle, la carte est centrée sur la ROI et affichée.

In [16]:
m2 = geemap.Map()

for year, comp in results.items():
    if comp is None:
        print(f"Aucun composite pour {year} (skip)")
        continue

    satellite = satellite_by_year.get(year)
    if satellite is None:
        print(f"Satellite non défini pour {year} (skip)")
        continue

    cfg = config[satellite]

    rgb_bands, nrg_bands = ['R', 'G', 'B'], ['N', 'R', 'G']
    vis_rgb = { 'bands': rgb_bands, 'min': 0.0, 'max': 0.3, 'gamma': 1.2 }
    vis_nrg = { 'bands': nrg_bands, 'min': 0.0, 'max': 0.3, 'gamma': 1.2 }

    m2.addLayer(comp, vis_rgb, f"L{satellite}-{year}-RGB")
    m2.addLayer(comp, vis_nrg, f"L{satellite}-{year}-NRG")

m2.center_object(ROI_ee, zoom=13)
m2

Map(center=[16.039745167717093, -16.45725266033479], controls=(WidgetControl(options=['position', 'transparent…

<img src="images/dowloaded_img_viz.png" width="2000">